In [2]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")          # trabalhar sempre a partir da raiz do projeto

import duckdb
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

con = duckdb.connect() #data base on memory

con.sql("""
CREATE OR REPLACE VIEW survey AS
SELECT * FROM 'data/processed/survey_clean.parquet'
""")

con.sql("Describe survey").df()[["column_name", "column_type"]]

,column_name,column_type
0,response_id,BIGINT
1,employment,VARCHAR
2,country,VARCHAR
3,dev_type,VARCHAR
4,ed_level,VARCHAR
5,org_size,VARCHAR
6,remote_work,VARCHAR
7,years_pro,DOUBLE
8,years_pro_alt,VARCHAR
9,years_code,DOUBLE


In [3]:
con.sql("SELECT YEAR, COUNT(*) AS n FROM survey GROUP BY year ORDER BY year").df()

,year,n
0,2024,20669
1,2025,18513


## AI adoption per year

In [4]:
query = """
SELECT year, 
    COUNT(*) AS n_total, 
    COUNT(uses_ai) AS n_responses, 
    ROUND(100 * AVG(uses_ai::INT),1) As pct_ai_use
FROM survey
GROUP BY year
Order By Year
"""

con.sql(query).df()

,year,n_total,n_responses,pct_ai_use
0,2024,20669,20669,62.6
1,2025,18513,18037,80.8


In [5]:

d1 = con.sql(query).df()
d1["year"] = d1["year"].astype(str) 
fig = px.bar(d1, x="year", y="pct_ai_use", text="pct_ai_use",
             title="AI adoption between developers workers (2024 vs 2025)",
             labels={"pct_ai_use": "% that use ai", "year": ""})
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_yaxes(range=[0, 100])
fig.show()

## Frequency of AI use in 2025

In [6]:
q_frequency = """
SELECT ai_use_frequency,
    COUNT(*) AS n, 
    COUNT(uses_ai) AS n_responses, 
    ROUND(100 * COUNT(*) /SUM(COUNT(*)) OVER (), 1) AS pct
FROM survey
WHERE year = 2025 AND uses_ai
GROUP BY ai_use_frequency
ORDER BY n DESC
"""

d2 = con.sql(q_frequency).df()
d2

,ai_use_frequency,n,n_responses,pct
0,daily,8721,8721,59.9
1,weekly,3386,3386,23.2
2,monthly_or_less,2464,2464,16.9


In [7]:
fig = px.bar(d2, x="pct", y="ai_use_frequency", orientation="h", text="pct",
             title="How often do developers use AI (2025, users only)",
             labels={"pct": "% of users", "ai_use_frequency": ""},
             category_orders={"ai_use_frequency": ["daily", "weekly", "monthly_or_less"]})
fig.update_traces(texttemplate="%{text}%")
fig.update_yaxes(autorange="reversed")
fig.show()

## Adoption by country and the decision on the minimum sample size.

In [8]:
query_countries ="""
SELECT country,
        Count(*) FILTER (WHERE year = 2024) AS n_2024,
        COUNT(*) FILTER (WHERE year = 2025) AS n_2025,
FROM survey
WHERE uses_ai IS NOT NULL
GROUP BY country
ORDER BY LEAST(n_2024, n_2025) DESC
LIMIT 25
"""

con.sql(query_countries).df()

,country,n_2024,n_2025
0,United States of America,4306,4220
1,Germany,1918,1749
2,United Kingdom of Great Britain and Northern I...,1260,1223
3,India,923,802
4,France,830,781
5,Canada,790,721
6,Netherlands,502,500
7,Brazil,615,456
8,Italy,481,444
9,Spain,509,438


In [9]:
q_limiares = """
WITH por_pais AS(
    SELECT country,
         COUNT(*) FILTER (WHERE year = 2024) AS n_2024,
         COUNT(*) FILTER (WHERE year = 2025) AS n_2025
  FROM survey
  WHERE uses_ai IS NOT NULL
  GROUP BY country
)
SELECT t.limiar,
        COUNT(*) FILTER (WHERE LEAST(n_2024, n_2025) >= t.limiar) AS countries_who_pass
FROM por_pais
CROSS JOIN (VALUES (30), (50), (100), (200)) AS t(limiar)
GROUP BY t.limiar
ORDER BY t.limiar
"""

con.sql(q_limiares).df()

,limiar,countries_who_pass
0,30,58
1,50,46
2,100,31
3,200,17


In [10]:
q_cobertura = """
WITH por_pais AS (
  SELECT country,
         COUNT(*) FILTER (WHERE year = 2024) AS n_2024,
         COUNT(*) FILTER (WHERE year = 2025) AS n_2025
  FROM survey WHERE uses_ai IS NOT NULL GROUP BY country
)
SELECT t.limiar,
       COUNT(*) FILTER (WHERE LEAST(n_2024, n_2025) >= t.limiar) AS paises,
       ROUND(100.0 * SUM(n_2024 + n_2025) FILTER (WHERE LEAST(n_2024, n_2025) >= t.limiar)
             / SUM(n_2024 + n_2025), 1) AS pct_respostas_cobertas
FROM por_pais CROSS JOIN (VALUES (30), (50), (100), (200)) AS t(limiar)
GROUP BY t.limiar ORDER BY t.limiar
"""
con.sql(q_cobertura).df()

,limiar,paises,pct_respostas_cobertas
0,30,58,95.9
1,50,46,92.9
2,100,31,86.7
3,200,17,75.6


In [11]:
MIN_N = 100

q_pais = f"""
WITH base AS (
  SELECT country, year,
         COUNT(uses_ai)    AS n,
         SUM(uses_ai::INT) AS n_usa
  FROM survey
  GROUP BY country, year
),
validos AS (
  SELECT country
  FROM base
  GROUP BY country
  HAVING COUNT(*) = 2 AND MIN(n) >= {MIN_N}
)
SELECT b.country, b.year, b.n, b.n_usa,
       ROUND(100.0 * b.n_usa / b.n, 1) AS pct_usa_ia
FROM base b
JOIN validos v USING (country)
ORDER BY b.country, b.year
"""
d_pais = con.sql(q_pais).df()
d_pais


,country,year,n,n_usa,pct_usa_ia
0,Australia,2024,466,262.0,56.2
1,Australia,2025,437,337.0,77.1
2,Austria,2024,310,200.0,64.5
3,Austria,2025,234,191.0,81.6
4,Belgium,2024,172,108.0,62.8
...,...,...,...,...,...
57,Ukraine,2025,430,399.0,92.8
58,United Kingdom of Great Britain and Northern I...,2024,1260,685.0,54.4
59,United Kingdom of Great Britain and Northern I...,2025,1223,879.0,71.9
60,United States of America,2024,4306,2432.0,56.5


In [12]:
nomes = {"United States of America": "United States",
         "United Kingdom of Great Britain and Northern Ireland": "United Kingdom"}
d = d_pais.copy()
d["country"] = d["country"].replace(nomes)

w = d.pivot(index="country", columns="year", values="pct_usa_ia").reset_index()
w = w.sort_values(2025)

fig = go.Figure()
for _, r in w.iterrows():
    fig.add_trace(go.Scatter(x=[r[2024], r[2025]], y=[r["country"]] * 2, mode="lines",
                             line=dict(color="lightgray", width=3),
                             showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=w[2024], y=w["country"], mode="markers", name="2024", marker=dict(size=10)))
fig.add_trace(go.Scatter(x=w[2025], y=w["country"], mode="markers", name="2025", marker=dict(size=10)))
fig.update_layout(title=f"Adoção de IA por país (n ≥ {MIN_N} por ano)",
                  xaxis_title="% that uses Ai", height=max(350, 22 * len(w)))
fig.show()

In [13]:
w = d.pivot(index="country", columns="year", values="pct_usa_ia").reset_index()
w["delta_pp"] = (w[2025] - w[2024]).round(1)
w = w.sort_values("delta_pp", ascending=False)

print(w[["country", 2024, 2025, "delta_pp"]].head(5))   # maiores subidas
print(w[["country", 2024, 2025, "delta_pp"]].tail(5))   # menores subidas

year      country  2024  2025  delta_pp
10         Greece  67.1  91.8      24.7
18    New Zealand  56.3  80.9      24.6
22        Romania  63.5  87.6      24.1
15          Italy  58.4  82.0      23.6
7         Finland  58.3  81.8      23.5
year       country  2024  2025  delta_pp
6          Denmark  63.2  77.1      13.9
27          Turkey  72.4  86.0      13.6
25          Sweden  68.0  80.1      12.1
19          Norway  67.1  78.7      11.6
23    South Africa  74.5  84.8      10.3


## Adoption by professional function

In [14]:
q_function =f"""
WITH longo AS (
    SELECT year, uses_ai,
            TRIM(unnest(string_split(dev_type,';'))) AS function
    FROM survey
    WHERE dev_type Is NOT NULL
),
base AS (
    SELECT function, year,
            COUNT(uses_ai) AS n,
            SUM(uses_ai::INT)::INT AS n_usa
    FROM longo
    GROUP BY function, year
)
SELECT function, year, n, n_usa, ROUND(100.0 * n_usa / n, 1) as pct_use_ai
FROM base
WHERE function IN (SELECT function FROM base GROUP BY function HAVING COUNT(*) = 2 AND MIN(n) >= {MIN_N})
ORDER BY function, year
"""

d_func = con.sql(q_function).df()
d_func


,function,year,n,n_usa,pct_use_ai
0,Academic researcher,2024,368,197,53.5
1,Academic researcher,2025,373,263,70.5
2,Cloud infrastructure engineer,2024,253,159,62.8
3,Cloud infrastructure engineer,2025,216,185,85.6
4,Data engineer,2024,509,306,60.1
5,Data engineer,2025,397,337,84.9
6,Data or business analyst,2024,182,111,61.0
7,Data or business analyst,2025,135,90,66.7
8,"Developer, QA or test",2024,216,132,61.1
9,"Developer, QA or test",2025,156,130,83.3


In [15]:
def dumbbell(d, label, title, height_per_row=22):
    w = d.pivot(index=label, columns="year", values="pct_use_ai").reset_index().sort_values(2025)
    fig = go.Figure()
    for _, r in w.iterrows():
        fig.add_trace(go.Scatter(x=[r[2024], r[2025]], y=[r[label]] * 2, mode="lines",
                                 line=dict(color="lightgray", width=3),
                                 showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=w[2024], y=w[label], mode="markers", name="2024", marker=dict(size=10)))
    fig.add_trace(go.Scatter(x=w[2025], y=w[label], mode="markers", name="2025", marker=dict(size=10)))
    fig.update_layout(title=title, xaxis_title="% que usa IA",
                      height=max(350, height_per_row * len(w)))
    return fig, w

fig, w_func = dumbbell(d_func, "function", f"Adoção de IA por função (n ≥ {MIN_N} por ano)")
fig.show()

w_func["delta_pp"] = (w_func[2025] - w_func[2024]).round(1)
w_func.sort_values("delta_pp", ascending=False)[["function", 2024, 2025, "delta_pp"]]

year,function,2024,2025,delta_pp
14,Project manager,53.0,82.7,29.7
2,Data engineer,60.1,84.9,24.8
6,"Developer, desktop or enterprise applications",46.5,70.9,24.4
12,Engineering manager,62.6,85.5,22.9
1,Cloud infrastructure engineer,62.8,85.6,22.8
4,"Developer, QA or test",61.1,83.3,22.2
7,"Developer, embedded applications or devices",45.3,65.9,20.6
15,System administrator,43.6,64.1,20.5
11,"Developer, mobile",65.6,85.5,19.9
5,"Developer, back-end",62.9,81.8,18.9


In [17]:
q_exp= """
WITH base AS(
    SELECT year, uses_ai,
        CASE WHEN years_pro < 2 THEN 1
            WHEN years_pro < 5 THEN 2
            WHEN years_pro < 10 THEN 3
            WHEN years_pro < 20 THEN 4
            ELSE 5 END AS ordem,
        CASE WHEN years_pro < 2 THEN '0-1 anos'
            WHEN years_pro < 5 THEN '2-4 anos'
            WHEN years_pro < 10 THEN '5-9 anos'
            WHEN years_pro < 20 THEN '10-19 anos'
            ELSE '20+ anos' END AS faixa
    FROM survey
    WHERE years_pro IS NOT NULL    
)

SELECT faixa, year, COUNT(uses_ai) AS n, SUM(uses_ai::INT)::INT AS n_usa,
        ROUND(100.0 * SUM(uses_ai::INT) / COUNT(uses_ai), 1) As pct_usa_ia
FROM base
GROUP BY ordem, faixa, year
ORDER BY ordem, year
"""

d_exp = con.sql(q_exp).df()
d_exp

,faixa,year,n,n_usa,pct_usa_ia
0,0-1 anos,2024,1251,914,73.1
1,0-1 anos,2025,556,471,84.7
2,2-4 anos,2024,4428,3146,71.0
3,2-4 anos,2025,2667,2264,84.9
4,5-9 anos,2024,5696,3667,64.4
5,5-9 anos,2025,4281,3550,82.9
6,10-19 anos,2024,5880,3474,59.1
7,10-19 anos,2025,6084,4900,80.5
8,20+ anos,2024,3414,1747,51.2
9,20+ anos,2025,4449,3386,76.1


In [21]:

d = d_exp.copy()
d["ano"] = d["year"].astype(str)          # ano como texto, para ser categoria
ordem = ["0-1 years", "2-4 years", "5-9 years", "10-19 years", "20+ years"]

fig = px.bar(
    d, x="faixa", y="pct_usa_ia", color="ano", barmode="group", text="pct_usa_ia",
    category_orders={"faixa": ordem},
    hover_data={"n": True, "n_usa": True},
    title="AI adoption by years of professional experience",
    labels={"pct_usa_ia": "% that use AI", "faixa": "Experience", "ano": "year"},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_yaxes(range=[0, 100])
fig.show()

In [22]:
q_sal = """
SELECT year,
       COUNT(*)                                AS n,
       ROUND(AVG(comp_yearly))                 AS media,
       ROUND(MEDIAN(comp_yearly))              AS mediana,
       ROUND(QUANTILE_CONT(comp_yearly, 0.25)) AS q1,
       ROUND(QUANTILE_CONT(comp_yearly, 0.75)) AS q3
FROM survey
GROUP BY year
ORDER BY year
"""
con.sql(q_sal).df()

,year,n,media,mediana,q1,q3
0,2024,20669,79989.0,66592.0,36798.0,107406.0
1,2025,18513,94301.0,81210.0,46986.0,125000.0


In [23]:


d_sal = con.sql("SELECT year::VARCHAR AS ano, comp_yearly FROM survey").df()

fig = px.box(d_sal, x="ano", y="comp_yearly", log_y=True, points=False,
             labels={"comp_yearly": "Remuneração anual (USD, escala log)", "ano": "Ano"},
             title="Distribuição da remuneração anual por ano")
fig.show()